# Analysis 19 — who writes like whom?

Character-trigram counts on equal windows of each author's text (log: `analysis-log.md`, Analysis 19).

Choices so far (student): messages glued per author with a line break, cut into 1,500-character
windows, the same number of windows per author, links removed (2026-10-27; was a `<URL>` placeholder), capitals kept.
Stage 3 option C: raw counts first, then read back which trigrams drive the distances before
deciding whether to drop or standardise anything.

In [ ]:
import re

import numpy as np
import pandas as pd
from goad_toolkit.datatransforms import Pipeline, TransformBase
from sklearn.feature_extraction.text import CountVectorizer

from wa_analyzer.data import load_own_chat

SEED = 42
WINDOW = 1_500  # characters per window (student)
SEPARATOR = "\n"  # between messages; Stage 6 reruns with " "
GLUED_SYSTEM_LINE = re.compile(r"\d{2}-\d{2}-\d{4} \d{2}:\d{2} - ")  # parser glitch; carries real names
NOT_A_TEXT = re.compile(r"<Media weggelaten>|<Video note omitted>|bericht is verwijderd|locatie gedeeld|^locatie: |\.vcf \(bestand")
EDIT_MARKER = re.compile(r"\s*<Dit bericht is bewerkt>")  # added by WhatsApp, not typed
URL = re.compile(r"\s*(?:https?://\S+|www\.\S+)")  # with the space before it

## Clean

The lesson's `NormaliseText` lowercases and collapses all whitespace, which would undo two of the
choices above (capitals, line breaks). This step keeps both and only strips links and WhatsApp's edit marker.

In [ ]:
class CleanForStyle(TransformBase):
    """Drop rows that are not typed text, and strip WhatsApp's edit marker and links."""

    def transform(self, data: pd.DataFrame, column: str, feature: str = "text") -> pd.DataFrame:
        raw = data[column].fillna("").astype(str)
        text = raw.str.replace(EDIT_MARKER, "", regex=True).str.replace(URL, "", regex=True)
        typed = ~raw.str.contains(GLUED_SYSTEM_LINE) & ~raw.str.contains(NOT_A_TEXT) & (text.str.strip() != "")  # link-only messages drop out
        data = data[typed].copy()
        data[feature] = text[typed]
        return data


own = load_own_chat().sort_values(["author", "timestamp"])
clean = Pipeline().add(CleanForStyle, column="message").apply(own)
print(f"{len(clean):,} of {len(own):,} messages kept")

## Windows

Each author's messages, in time order, glued into one text and cut at exactly `WINDOW` characters
(a window may start or end mid-message). The leftover at the end is dropped. Every author then
gets the same number of windows, drawn at random from all of theirs.

In [ ]:
def windows_per_author(frame: pd.DataFrame, size: int, separator: str) -> pd.DataFrame:
    """Cut each author's glued text into consecutive windows of exactly `size` characters."""
    rows = []
    for author, messages in frame.groupby("author"):
        glued = separator.join(messages.text)
        starts = np.cumsum([0] + [len(t) + len(separator) for t in messages.text[:-1]])  # where each message begins
        times = pd.to_datetime(messages.timestamp).to_numpy()
        for i in range(0, len(glued) - size + 1, size):
            first, last = np.searchsorted(starts, [i, i + size - 1], side="right") - 1
            rows.append({"author": author, "text": glued[i : i + size],
                         "middle": times[first] + (times[last] - times[first]) / 2})
    return pd.DataFrame(rows)


every_window = windows_per_author(clean, WINDOW, SEPARATOR)
per_author = every_window.author.value_counts()
n_each = per_author.min()
windows = every_window.groupby("author").sample(n_each, random_state=SEED).reset_index(drop=True)
print(f"windows per author before balancing:\n{per_author.to_string()}")
print(f"\nbalanced: {n_each} windows x {windows.author.nunique()} authors = {len(windows)}")

## Counts

`lowercase=False`: `CountVectorizer` lowercases by default, which would throw the capitals away.

In [ ]:
vectorizer = CountVectorizer(analyzer="char", ngram_range=(3, 3), lowercase=False)
counts = vectorizer.fit_transform(windows.text).toarray()
trigrams = vectorizer.get_feature_names_out()
print(f"{counts.shape[0]} windows x {counts.shape[1]:,} trigrams; a window uses {np.mean(counts > 0):.1%} of them")

## Which trigrams drive the distance? (option C)

The Manhattan distance between two windows is a sum over trigrams of |count difference|. Adding that
up over every pair of windows gives each trigram's share of all the distance in the data.
Sorting a column makes the pairwise sum cheap: the k-th smallest of n values appears with a plus sign
k−1 times and a minus sign n−k times.

In [ ]:
def pairwise_abs_sum(column_counts: np.ndarray) -> np.ndarray:
    """Per column: the sum of |x_i - x_j| over all pairs of rows."""
    n = column_counts.shape[0]
    weights = 2 * np.arange(1, n + 1) - n - 1
    return weights @ np.sort(column_counts, axis=0)


contribution = pairwise_abs_sum(counts)
readback = pd.DataFrame({
    "trigram": [repr(t) for t in trigrams],
    "share_of_distance": contribution / contribution.sum(),
    "mean_count": counts.mean(axis=0),
    "windows_with_it": (counts > 0).mean(axis=0),
}).sort_values("share_of_distance", ascending=False).reset_index(drop=True)

cumulative = readback.share_of_distance.cumsum()
for top in (10, 50, 200):
    print(f"top {top:>3} trigrams: {cumulative.iloc[top - 1]:.0%} of all distance")
print(f"trigrams needed for half of all distance: {int((cumulative < 0.5).sum()) + 1:,} of {len(readback):,}")
readback.head(25)

In [ ]:
rare = readback[readback.windows_with_it < 3 / len(windows)]
print(f"trigrams in fewer than 3 windows: {len(rare):,} ({len(rare) / len(readback):.0%} of all), "
      f"together {rare.share_of_distance.sum():.0%} of all distance")

## Step 1 — do the authors separate at all?

Silhouette on the Manhattan distances in the full trigram space, with the real author labels and
with shuffled ones (as `separation` in 06.2). If real is no better than shuffled, there is no style to group.

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import manhattan_distances

distance = manhattan_distances(counts, counts)
authors = windows.author.to_numpy()
rng = np.random.default_rng(SEED)
real = silhouette_score(distance, authors, metric="precomputed")
shuffled = [silhouette_score(distance, rng.permutation(authors), metric="precomputed") for _ in range(200)]
print(f"silhouette by author: {real:.3f}   shuffled labels: {np.mean(shuffled):.3f} "
      f"(highest of 200: {np.max(shuffled):.3f})")

# leave-one-out 1-NN: is each window's nearest other window by the same author? (chance = 1 in 9)
nearest = np.where(np.eye(len(distance), dtype=bool), np.inf, distance).argmin(axis=1)
print(f"1-NN, same author: {np.mean(authors[nearest] == authors):.0%}   (chance: {1 / len(set(authors)):.0%})")

## Stage 4 sketches

A: the obvious high-dimensional family — every window as a dot, coloured by author (PCA on the counts,
and t-SNE on the Manhattan distances). B: the alternative — one cell per pair of authors, the Manhattan
distance between their average windows (author centroids), so no projection in between.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

pca = PCA(n_components=2, random_state=SEED)
pca_xy = pca.fit_transform(counts)
tsne_xy = TSNE(n_components=2, metric="precomputed", init="random", perplexity=30,
               random_state=SEED).fit_transform(distance)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
palette = dict(zip(sorted(set(authors)), plt.get_cmap("tab10").colors))
for ax, xy, title in [
    (axes[0], pca_xy, f"PCA on counts ({pca.explained_variance_ratio_.sum():.0%} of variance in 2 components)"),
    (axes[1], tsne_xy, "t-SNE on Manhattan distances (perplexity 30)"),
]:
    for author, colour in palette.items():
        mask = authors == author
        ax.scatter(xy[mask, 0], xy[mask, 1], s=14, color=colour, label=author, alpha=0.8)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
axes[1].legend(fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False)
fig.suptitle("Sketch A — 306 windows of 1,500 characters, coloured by author")
fig.tight_layout()
fig.savefig("sketch-A-style-projection.png", dpi=120, bbox_inches="tight")

In [ ]:
centroids = pd.DataFrame(counts).groupby(authors).mean()
between = pd.DataFrame(manhattan_distances(centroids, centroids), index=centroids.index, columns=centroids.index)

fig, ax = plt.subplots(figsize=(7.5, 6.5))
shown = between.where(~np.eye(len(between), dtype=bool))
image = ax.imshow(shown, cmap="viridis_r")
ax.set_xticks(range(len(between)), between.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(between)), between.index, fontsize=8)
for i in range(len(between)):
    for j in range(len(between)):
        if i != j:
            ax.text(j, i, f"{between.iat[i, j]:.0f}", ha="center", va="center", fontsize=7, color="white")
fig.colorbar(image, ax=ax, label="Manhattan distance between average windows (lower = more alike)")
ax.set_title("Sketch B — how far apart is each pair of authors?", fontsize=10)
fig.tight_layout()
fig.savefig("sketch-B-style-author-distances.png", dpi=120, bbox_inches="tight")

## Step 2 — are authors closer to their own background group?

Groups fixed in Stage 1, read from a gitignored file (pseudonym + study background could identify people).
Statistic: average centroid distance between two authors in the *same* group minus the average between
authors in *different* groups (negative = groups hang together). Chance: the same statistic for every one
of the 9!/(3!·4!·2!) = 1,260 ways to split 9 authors into groups of 3, 4 and 2.

In [ ]:
from itertools import combinations

groups = pd.read_csv("../../data/processed/author-groups.csv").set_index("author").group
assert set(groups.index) == set(between.index), "every author needs a group"  # noqa: S101


def within_minus_between(distances: pd.DataFrame, labels: pd.Series) -> float:
    """Mean distance of same-group author pairs minus mean distance of different-group pairs."""
    same, different = [], []
    for a, b in combinations(distances.index, 2):
        (same if labels[a] == labels[b] else different).append(distances.at[a, b])
    return float(np.mean(same) - np.mean(different))


def every_split(names: list[str], sizes: list[int]):
    """Yield every way to put `names` into groups of the given sizes, as a Series name -> group number."""
    if not sizes:
        yield pd.Series(dtype=int)
        return
    for chosen in combinations(names, sizes[0]):
        rest = [n for n in names if n not in chosen]
        for tail in every_split(rest, sizes[1:]):
            yield pd.concat([pd.Series(len(sizes), index=list(chosen)), tail])


observed = within_minus_between(between, groups)
splits = [within_minus_between(between, split) for split in every_split(list(between.index), [3, 4, 2])]
as_low = np.mean(np.array(splits) <= observed)
print(f"{len(splits):,} splits; real split: {observed:+.1f} (same-group pairs minus different-group pairs)")
print(f"splits that hang together at least as well as the real one: {as_low:.0%}")
print(f"range over all splits: {min(splits):+.1f} to {max(splits):+.1f}")

for name, members in groups.groupby(groups).groups.items():
    pairs = [between.at[a, b] for a, b in combinations(members, 2)]
    print(f"  {name:<10} ({len(members)} people): mean distance within {np.mean(pairs):.0f}")

## Perplexity sweep (6.6)

Colour = background group, marker = author. What should survive a sweep is which windows sit together;
positions, sizes and gaps will move.

In [ ]:
MARKERS = ["o", "s", "^", "D"]
GROUP_COLOURS = dict(zip(sorted(groups.unique()), plt.get_cmap("Set1").colors))
style = {}
for name, members in groups.groupby(groups).groups.items():
    for marker, author in zip(MARKERS, sorted(members)):
        style[author] = (GROUP_COLOURS[name], marker)

PERPLEXITIES = [5, 15, 30, 50]
fig, axes = plt.subplots(2, 2, figsize=(12, 11))
for ax, perplexity in zip(axes.flat, PERPLEXITIES):
    xy = TSNE(n_components=2, metric="precomputed", init="random", perplexity=perplexity,
              random_state=SEED).fit_transform(distance)
    for author, (colour, marker) in style.items():
        mask = authors == author
        ax.scatter(xy[mask, 0], xy[mask, 1], s=18, color=colour, marker=marker, alpha=0.8,
                   label=f"{groups[author]}: {author}")
    ax.set_title(f"perplexity {perplexity}", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
handles, labels_ = axes.flat[0].get_legend_handles_labels()
order = np.argsort(labels_)
fig.legend([handles[i] for i in order], [labels_[i] for i in order], loc="lower center", ncol=3,
           fontsize=8, frameon=False)
fig.suptitle("Sketch C — t-SNE perplexity sweep, coloured by background group")
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig("sketch-C-style-perplexity-sweep.png", dpi=110, bbox_inches="tight")

## Draft

Perplexity 15 (student): the per-person clumps are clear there. Colour = background group (validated palette,
colour-blind safe), shape = author. The numbers in the subtitle come from the full trigram space, not the picture.

In [ ]:
DRAFT_PERPLEXITY = 15
DRAFT_COLOURS = {"economics": "#2a78d6", "technical": "#eb6834", "education": "#1baf7a"}
same_author_share = np.mean(authors[nearest] == authors)

xy = TSNE(n_components=2, metric="precomputed", init="random", perplexity=DRAFT_PERPLEXITY,
          random_state=SEED).fit_transform(distance)

fig, ax = plt.subplots(figsize=(10, 8))
for group_name in DRAFT_COLOURS:
    members = sorted(groups[groups == group_name].index)
    for marker, author in zip(MARKERS, members):
        mask = authors == author
        ax.scatter(xy[mask, 0], xy[mask, 1], s=34, marker=marker, color=DRAFT_COLOURS[group_name],
                   edgecolor="white", linewidth=0.6, label=f"{group_name}: {author}")
ax.set_xticks([])
ax.set_yticks([])
for side in ax.spines.values():
    side.set_color("#c9c8c2")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=9,
          title="study background: person", title_fontsize=9, alignment="left")
fig.suptitle("Group members have their own writing style,\nbut study background does not predict that style",
             x=0.02, ha="left", fontsize=14, fontweight="bold")
ax.set_title(
    f"Each dot is 1,500 characters of one person's messages. A dot's nearest neighbour is by the same person "
    f"{same_author_share:.0%} of the time (chance {1 / len(set(authors)):.0%}),\n"
    f"but people in the same study group are no closer to each other than in a random split "
    f"({as_low:.0%} of all 1,260 splits do as well).",
    loc="left", fontsize=9, color="#555550",
)
fig.text(0.02, 0.01,
         f"9 people, 34 windows each, counted as character trigrams. t-SNE (perplexity {DRAFT_PERPLEXITY}) keeps who sits "
         "together, not distances: clump sizes and gaps mean nothing.\nPer-person clumps also appear at perplexity "
         "5, 30 and 50. Numbers computed on the full counts, not on this picture.",
         fontsize=8, color="#555550", ha="left")
fig.tight_layout(rect=(0, 0.05, 1, 0.97))
plt.show()  # draft 1 was handed in on 23 Oct; its png stays as submitted, so this cell no longer writes it

## Stage 6 — check A1: style or conversation?

Prediction written down first (student): 61% drops to about 50%; below 30% the title changes.
Each window may only match windows whose middle date is more than half a year away, so neighbouring
windows from the same conversation can't be the match. The rule is the same for every author.

In [ ]:
MIN_GAP = pd.Timedelta(days=182)

middles = windows.middle.to_numpy()
too_close = np.abs(middles[:, None] - middles[None, :]) <= MIN_GAP.to_timedelta64()
far_nearest = np.where(too_close, np.inf, distance).argmin(axis=1)
candidates = (~too_close).sum(axis=1)
own_candidates = np.array([np.sum(~too_close[i] & (authors == authors[i])) for i in range(len(authors))])
far_chance = np.mean(own_candidates / candidates)
print(f"1-NN, same author, all windows:          {same_author_share:.0%}")
print(f"1-NN, same author, > half a year apart:  {np.mean(authors[far_nearest] == authors):.0%}   "
      f"(chance with this rule: {far_chance:.0%})")
print(f"windows with no own-author window that far away: {np.sum(own_candidates == 0)}")

## Stage 6 — second grouping: lives in the city or not

Thought of after the draft, so it is comparison 2 of 2 and reported as such. Fixed before running (student):
city people write more alike (shorter messages, more laughing, more emoji); a finding only if fewer than 5%
of the 9!/(5!·4!) = 126 splits hang together at least as well.

In [ ]:
lives = pd.read_csv("../../data/processed/author-city.csv").set_index("author").lives
assert set(lives.index) == set(between.index), "every author needs a value"  # noqa: S101

city_observed = within_minus_between(between, lives)
city_splits = [within_minus_between(between, split) for split in every_split(list(between.index), [5, 4])]
city_as_low = np.mean(np.array(city_splits) <= city_observed)
print(f"{len(city_splits)} splits; real split: {city_observed:+.1f}")
print(f"splits that hang together at least as well as the real one: {city_as_low:.0%}")
for name, members in lives.groupby(lives).groups.items():
    pairs = [between.at[a, b] for a, b in combinations(members, 2)]
    print(f"  {name:<9} ({len(members)} people): mean distance within {np.mean(pairs):.0f}")

## Story — what is each person's clump made of?

Predictions written down first (student): animated-elk sends short messages and little "haha"; pliable-tiger
uses many emoji and smileys; vibrant-barracuda asks many questions. Four direct measures check those, then per
author the trigrams whose average count differs most from the other eight. This describes the clumps; it tests
no new grouping.

In [ ]:
LAUGH = re.compile(r"ha(?:ha)+h?", re.IGNORECASE)
EMOJI_OR_SMILEY = re.compile("[\U0001F300-\U0001F3FA\U0001F400-\U0001FAFF\u2600-\u27BF]|[:;]-?[()DPp]")  # not raw: Arrow regex has no \\U; skin tones skipped
QUESTION = re.compile(r"\?")


def per_thousand(texts: pd.Series, pattern: re.Pattern) -> float:
    """Matches of `pattern` per 1,000 characters of the author's text."""
    return 1_000 * texts.str.count(pattern).sum() / texts.str.len().sum()


habits = pd.DataFrame({
    "median message length": clean.groupby("author").text.apply(lambda t: t.str.len().median()),
    "haha per 1,000": windows.groupby("author").text.apply(per_thousand, pattern=LAUGH),
    "emoji/smiley per 1,000": windows.groupby("author").text.apply(per_thousand, pattern=EMOJI_OR_SMILEY),
    "? per 1,000": windows.groupby("author").text.apply(per_thousand, pattern=QUESTION),
})
ranks = habits.rank(ascending=False).astype(int)  # 1 = highest of the nine
print(habits.round(1).astype(str).add(" (#").add(ranks.astype(str)).add(")").to_string())

In [ ]:
TOP = 8
author_means = pd.DataFrame(counts, columns=trigrams).groupby(authors).mean()

for author in author_means.index:
    difference = author_means.loc[author] - author_means.drop(author).mean()
    top_share = difference.abs().nlargest(2 * TOP).sum() / difference.abs().sum()
    more = ", ".join(f"{t!r} {d:+.1f}" for t, d in difference.nlargest(TOP).items())
    less = ", ".join(f"{t!r} {d:+.1f}" for t, d in difference.nsmallest(TOP).items())
    print(f"{author}  (these {2 * TOP} trigrams: {top_share:.0%} of the distance to the rest)")
    print(f"  more than the rest: {more}")
    print(f"  less than the rest: {less}\n")

### Is hypnotic-rabbit's writing formal?

Predictions first (student): hypnotic-rabbit ranks in the top 3 for messages starting with a capital and for polite
words, #1 for full words (fewest chat short forms), #9 for emoji (that one is already in the table above).
Measured on all cleaned messages. A capital at the start may be the phone's autocapitalisation (check A3).

In [ ]:
CHAT_SHORT_FORMS = {"k", "nie", "ff", "idd", "mss", "wss", "ws", "gwn", "tog", "iig", "jwz", "thx", "w8", "dr", "t", "n", "m", "z"}
POLITE = {"helaas", "graag", "bedankt", "dank", "dankje", "dankjewel", "alsjeblieft", "alstublieft", "sorry"}
WORD = re.compile(r"[a-zà-ÿ0-9]+")


def formality(texts: pd.Series) -> pd.Series:
    """Three countable signs of formal writing in one author's messages."""
    first_letters = texts.str.extract(r"^\W*([^\W\d_])", expand=False).dropna()
    words = [w for t in texts.str.lower() for w in WORD.findall(t)]
    return pd.Series({
        "% starting with capital": 100 * first_letters.str.isupper().mean(),
        "chat short forms per 1,000 words": 1_000 * sum(w in CHAT_SHORT_FORMS for w in words) / len(words),
        "polite words per 1,000 words": 1_000 * sum(w in POLITE for w in words) / len(words),
    })


formal = clean.groupby("author").text.apply(formality).unstack()
formal_ranks = formal.rank(ascending=False).astype(int)  # 1 = highest; for short forms, #9 = most full words
print(formal.round(1).astype(str).add(" (#").add(formal_ranks.astype(str)).add(")").to_string())

## PCA: scree, and what PC1 and PC2 are made of

The Stage 4 PCA sketch kept 9% of the variance in 2 components. The scree shows how the variance is spread over
the components; the loadings show which trigrams each component weighs most (sign: which way along the axis).

In [ ]:
pca_full = PCA(n_components=50, random_state=SEED).fit(counts)
explained = pca_full.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 51), 100 * explained, color="#2a78d6")
ax.plot(range(1, 51), 100 * np.cumsum(explained), color="#eb6834", marker=".", label="cumulative")
ax.set_xlabel("component")
ax.set_ylabel("% of variance")
ax.set_title("Scree: variance per PCA component (raw trigram counts)")
ax.legend(frameon=False)
plt.show()
print("first 5 components:", ", ".join(f"{100 * e:.1f}%" for e in explained[:5]))
print(f"components needed for 50% of variance: {np.searchsorted(np.cumsum(explained), 0.5) + 1 if np.cumsum(explained)[-1] >= 0.5 else '> 50'}")

for k in range(2):
    loading = pd.Series(pca_full.components_[k], index=trigrams)
    print(f"\nPC{k + 1} ({100 * explained[k]:.1f}%)")
    print("  + side:", ", ".join(f"{t!r}" for t in loading.nlargest(10).index))
    print("  - side:", ", ".join(f"{t!r}" for t in loading.nsmallest(10).index))
    score = pd.Series(pca_full.transform(counts)[:, k], index=authors).groupby(level=0).mean().sort_values()
    print("  author means along it:", ", ".join(f"{a} {s:+.1f}" for a, s in score.items()))

## Does hypnotic-rabbit stand out more than the others?

Predictions first (student): hypnotic-rabbit is top 2 of 9 on distance to the rest and top 2 on the tightest clump.
All in the full trigram space. Caveat: their 34 windows are all their windows (back to back), the others a random 34
of more, so own-match and tightness are also shown with matches more than half a year apart only.

In [ ]:
def mean_pairwise(rows: np.ndarray, cols: np.ndarray, allowed: np.ndarray) -> float:
    """Mean window-to-window distance over the allowed pairs, without a window paired with itself."""
    block = distance[np.ix_(rows, cols)]
    keep = allowed[np.ix_(rows, cols)] & ~np.eye(len(distance), dtype=bool)[np.ix_(rows, cols)]
    return float(block[keep].mean())


anywhere = np.ones_like(distance, dtype=bool)
standing = pd.DataFrame({
    "distance to the rest": {a: mean_pairwise(authors == a, authors != a, anywhere) for a in between.index},
    "tightness (own windows)": {a: mean_pairwise(authors == a, authors == a, anywhere) for a in between.index},
    "tightness, > half a year": {a: mean_pairwise(authors == a, authors == a, ~too_close) for a in between.index},
    "own match %": pd.Series(100 * (authors[nearest] == authors), index=authors).groupby(level=0).mean(),
    "own match %, > half a year": pd.Series(100 * (authors[far_nearest] == authors), index=authors).groupby(level=0).mean(),
})
higher_is_more = {"distance to the rest": False, "tightness (own windows)": True, "tightness, > half a year": True,
                  "own match %": False, "own match %, > half a year": False}  # rank 1 = stands out most
standing_ranks = pd.DataFrame({c: standing[c].rank(ascending=higher_is_more[c]).astype(int) for c in standing})
print(standing.round(0).astype(int).astype(str).add(" (#").add(standing_ranks.astype(str)).add(")").to_string())

### What does hypnotic-rabbit write about? (words, not trigrams)

Predictions first (student): "kan", "niet", "ik", "helaas" near the top. Whole words, lowercased, over all cleaned
messages; difference in uses per 1,000 words against the other eight together (same idea as the trigram readback).
Check the lists for names, places or an employer before anything goes on a graph.

In [ ]:
# output not committed: the word list can contain names of real people; run locally to see it
WORDS = re.compile(r"[a-zà-ÿ]+")
words = clean.assign(word=clean.text.str.lower().str.findall(WORDS)).explode("word", ignore_index=True).dropna(subset=["word"])
word_counts = pd.crosstab(words.word, words.author)
FOCUS = "hypnotic-rabbit"

focus_rate = 1_000 * word_counts[FOCUS] / word_counts[FOCUS].sum()
rest = word_counts.drop(columns=FOCUS).sum(axis=1)
rest_rate = 1_000 * rest / rest.sum()
word_readback = pd.DataFrame({"per 1,000 (focus)": focus_rate, "per 1,000 (rest)": rest_rate,
                              "difference": focus_rate - rest_rate, "uses (focus)": word_counts[FOCUS]})
word_readback = word_readback[word_counts.sum(axis=1) >= 20]  # skip words too rare to compare
print(f"{FOCUS}: {word_counts[FOCUS].sum():,} words; rest: {rest.sum():,}")
print("\nmore than the rest:\n" + word_readback.nlargest(20, "difference").round(1).to_string())
print("\nless than the rest:\n" + word_readback.nsmallest(10, "difference").round(1).to_string())
for word in ["kan", "niet", "ik", "helaas"]:
    rank = int(word_readback.difference.rank(ascending=False)[word]) if word in word_readback.index else None
    print(f"{word!r}: rank {rank} of {len(word_readback):,} words on 'more than the rest'")

### Validate: are the planning words really hypnotic-rabbit's, and every year?

Prediction (student, stricter version): hypnotic-rabbit uses these words more than *each* of the other eight, not only
more than the eight pooled; same for messages that turn something down ("kan … niet" or "helaas"); and not just in
one period. The four first words were predicted before the word list was seen; "vanavond" and "gefeliciteerd" were
found in it, so for those two this is a description, not a test.

In [ ]:
CHECK_WORDS = ["ik", "niet", "kan", "helaas", "vanavond", "gefeliciteerd"]
DECLINE = re.compile(r"\bkan\b.*\bniet\b|\bhelaas\b", re.IGNORECASE)

per_person = (1_000 * word_counts.loc[CHECK_WORDS] / word_counts.sum()).T
per_person["declining msgs %"] = 100 * clean.text.str.contains(DECLINE).groupby(clean.author).mean()
person_ranks = per_person.rank(ascending=False).astype(int)
print("per 1,000 words (declining: % of messages); rank of 9 in brackets")
print(per_person.round(1).astype(str).add(" (#").add(person_ranks.astype(str)).add(")").to_string())

year = pd.to_datetime(clean.timestamp).dt.year
declining_by_year = clean.text.str.contains(DECLINE).groupby([year, clean.author]).mean().unstack() * 100
messages_by_year = clean.groupby([year, clean.author]).size().unstack()
focus_year_rank = declining_by_year.rank(axis=1, ascending=False)[FOCUS]
print("\ndeclining messages % per year: hypnotic-rabbit vs the rest")
print(pd.DataFrame({
    "hypnotic-rabbit %": declining_by_year[FOCUS].round(1),
    "rest median %": declining_by_year.drop(columns=FOCUS).median(axis=1).round(1),
    "rank of 9": focus_year_rank,
    "messages (focus)": messages_by_year[FOCUS],
}).to_string())

### Does the friend's clump hold at other perplexities?

Prediction (student): yes, at 5, 15, 30 and 50. The number per panel is measured on the picture itself: the share of the
friend's dots whose nearest dot on the picture is also theirs (in the full data it is 88%, 68% with half a year apart).

In [ ]:
FOCUS_COLOUR, CONTEXT_COLOUR = "#2a78d6", "#c3c2b7"
INK, SECONDARY_INK, MUTED_INK = "#0b0b0b", "#52514e", "#898781"
is_focus = authors == FOCUS

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, perplexity in zip(axes, PERPLEXITIES):
    sweep_xy = TSNE(n_components=2, metric="precomputed", init="random", perplexity=perplexity,
                    random_state=SEED).fit_transform(distance)
    on_picture = np.linalg.norm(sweep_xy[:, None] - sweep_xy[None], axis=-1)
    np.fill_diagonal(on_picture, np.inf)
    own_on_picture = np.mean(is_focus[on_picture[is_focus].argmin(axis=1)])
    ax.scatter(sweep_xy[~is_focus, 0], sweep_xy[~is_focus, 1], s=12, color=CONTEXT_COLOUR)
    ax.scatter(sweep_xy[is_focus, 0], sweep_xy[is_focus, 1], s=18, color=FOCUS_COLOUR)
    ax.set_title(f"perplexity {perplexity}: {own_on_picture:.0%} own neighbour", fontsize=10, color=INK)
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
fig.savefig("sketch-D-friend-perplexity-sweep.png", dpi=120)
plt.show()

## Stage 6 — how much do the numbers depend on which 34 windows are drawn?

Every author except the quietest has more than 34 windows, so `SEED` picks which 34 are used. The cleaning changes moved
the same-author score from 61% to 64% to 67%, partly through a different draw. Here the whole chain (draw, trigram counts,
distances, scores, group tests) is rerun for 20 draws, to report a range instead of one number.

In [ ]:
DRAWS = range(20)
author_names = sorted(every_window.author.unique())
study_splits = list(every_split(author_names, [3, 4, 2]))
city_splits_all = list(every_split(author_names, [5, 4]))


def scores_for_draw(seed: int) -> dict[str, float]:
    """All headline numbers for one random draw of n_each windows per author."""
    drawn = every_window.groupby("author").sample(n_each, random_state=seed).reset_index(drop=True)
    drawn_counts = CountVectorizer(analyzer="char", ngram_range=(3, 3), lowercase=False).fit_transform(drawn.text).toarray()
    drawn_distance = manhattan_distances(drawn_counts, drawn_counts)
    who = drawn.author.to_numpy()
    np.fill_diagonal(drawn_distance, np.inf)
    nearest_any = drawn_distance.argmin(axis=1)
    mids = drawn.middle.to_numpy()
    near_in_time = np.abs(mids[:, None] - mids[None, :]) <= MIN_GAP.to_timedelta64()
    nearest_far = np.where(near_in_time, np.inf, drawn_distance).argmin(axis=1)
    np.fill_diagonal(drawn_distance, 0)
    drawn_centroids = pd.DataFrame(drawn_counts).groupby(who).mean()
    drawn_between = pd.DataFrame(manhattan_distances(drawn_centroids, drawn_centroids),
                                 index=drawn_centroids.index, columns=drawn_centroids.index)
    study = np.array([within_minus_between(drawn_between, s) for s in study_splits])
    city = np.array([within_minus_between(drawn_between, s) for s in city_splits_all])
    to_rest = pd.Series({a: drawn_distance[np.ix_(who == a, who != a)].mean() for a in author_names})
    own_far = pd.Series(who[nearest_far] == who, index=who).groupby(level=0).mean()
    return {
        "same author %": 100 * np.mean(who[nearest_any] == who),
        "same author, > half a year %": 100 * np.mean(who[nearest_far] == who),
        "study splits as good %": 100 * np.mean(study <= within_minus_between(drawn_between, groups)),
        "city splits as good %": 100 * np.mean(city <= within_minus_between(drawn_between, lives)),
        "friend rank: distance to rest": int(to_rest.rank(ascending=False)[FOCUS]),
        "friend rank: own match > half a year": int(own_far.rank(ascending=False, method="min")[FOCUS]),
    }


draws = pd.DataFrame([scores_for_draw(seed) for seed in DRAWS], index=list(DRAWS))
summary = draws.describe().loc[["min", "50%", "max"]].T.rename(columns={"50%": "median"})
print(summary.round(0).astype(int).to_string())
print("\nfriend's rank on distance to the rest, over the draws:", draws["friend rank: distance to rest"].value_counts().sort_index().to_dict())
print("friend's rank on own match > half a year, over the draws:", draws["friend rank: own match > half a year"].value_counts().sort_index().to_dict())

## Draft 10 — the behaviour in the title, the panel as its evidence (after the seed range, which the footnote uses)

Draft 8 hid the point in the subtitle and a small panel. Draft 9 put the behaviour in the title and grouped the panel rows under the title words. Draft 10 (student's choice of words): courteous (thanks, graag, helaas) and practical (turning down plans) point right, casual (haha, idd, toch, lekker) points left; the title says "less casually than the rest of us", relative, because the numbers only show the others are *more* casual than this person, not that they are casual. Word ratios use the share of each person's words, against the median of the other eight.

Same t-SNE (perplexity 15), everyone grey except hypnotic-rabbit; no labels on the grey dots (their clumps don't show in grey, so names would only suggest positions). The label only carries what the full data backs:
2nd of 9 on distance to the rest, the words they use more and less (no names), laughing #9, polite words #1.
Title is a working title; the wording is the student's to choose.

In [ ]:
same_author_share = np.mean(authors[nearest] == authors)
focus_distance_rank = standing_ranks.at[FOCUS, "distance to the rest"]


def medoid(points: np.ndarray) -> np.ndarray:
    """The point with the smallest summed distance to the others: a label anchor inside the clump."""
    return points[np.abs(points[:, None] - points[None]).sum(axis=(1, 2)).argmin()]


xy = TSNE(n_components=2, metric="precomputed", init="random", perplexity=DRAFT_PERPLEXITY,
          random_state=SEED).fit_transform(distance)

fig = plt.figure(figsize=(13, 8.5))
ax = fig.add_axes((0.02, 0.09, 0.58, 0.74))  # fixed positions: scatter left, comparison panel right
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.scatter(xy[~is_focus, 0], xy[~is_focus, 1], s=30, color=CONTEXT_COLOUR, edgecolor="#fcfcfb", linewidth=0.6)
ax.scatter(xy[is_focus, 0], xy[is_focus, 1], s=46, color=FOCUS_COLOUR, edgecolor="#fcfcfb", linewidth=0.8)
focus_xy = xy[is_focus]
centre = medoid(focus_xy)
core = focus_xy[np.linalg.norm(focus_xy - centre, axis=1) <= 2 * np.median(np.linalg.norm(focus_xy - centre, axis=1))]
gap = 0.02 * np.ptp(xy[:, 0])
ax.text(core[:, 0].max() + gap, np.median(core[:, 1]), "first worker", ha="left", va="center", fontsize=10.5,
        fontweight="bold", color=INK)  # beside the clump; the stray dots are left out of the anchor
others = [a for a in author_names if a != FOCUS]
word_rates = word_counts / word_counts.sum()  # share of each person's words


def word_ratio(word: str) -> float:
    """How often the focus person uses `word`, as a multiple of the median of the other eight."""
    return word_rates.at[word, FOCUS] / word_rates.loc[word, others].median()


COURTEOUS, CASUAL = ["thanks", "graag", "helaas"], ["idd", "toch", "lekker"]  # casual: chosen by the student before reading the numbers
comparison = pd.Series({
    **{f"\u201c{w}\u201d": word_ratio(w) for w in COURTEOUS},
    "messages turning down plans": per_person.at[FOCUS, "declining msgs %"] / per_person.loc[others, "declining msgs %"].median(),
    "\u201chaha\u201d": habits.at[FOCUS, "haha per 1,000"] / habits.loc[others, "haha per 1,000"].median(),
    **{f"\u201c{w}\u201d": word_ratio(w) for w in CASUAL},
})
print(comparison.round(2).to_string())
panel = fig.add_axes((0.845, 0.27, 0.125, 0.52))  # right column, level with the top of the scatter
groups_in_title = {
    "courteous": [f"\u201c{w}\u201d" for w in COURTEOUS],
    "practical": ["messages turning down plans"],
    "casual": ["\u201chaha\u201d"] + [f"\u201c{w}\u201d" for w in CASUAL],
}
row_y, heading_y, y = {}, {}, 0.0
for heading, items in groups_in_title.items():  # top to bottom, in the title's word order
    heading_y[heading] = y
    for item in items:
        y -= 1
        row_y[item] = y
    y -= 1.4
rows = np.array([row_y[item] for item in comparison.index])
panel.axvline(1, color=SECONDARY_INK, linewidth=0.9)
panel.hlines(rows, 1, comparison.to_numpy(), color=FOCUS_COLOUR, linewidth=2.6)
panel.scatter(comparison.to_numpy(), rows, s=70, color=FOCUS_COLOUR, edgecolor="#fcfcfb", linewidth=0.9, zorder=3)
for row, ratio in zip(rows, comparison):
    panel.text(ratio * (1.3 if ratio > 1 else 0.77), row, f"{ratio:.1f}\u00d7", va="center",
               ha="left" if ratio > 1 else "right", fontsize=11, fontweight="bold", color=INK)
label_x = panel.get_yaxis_transform()  # x in panel fractions, y in rows
for heading, top in heading_y.items():
    panel.text(-1.12, top, heading, transform=label_x, ha="left", va="center", fontsize=11.5,
               fontweight="bold", color=FOCUS_COLOUR)
for item, row in row_y.items():
    panel.text(-1.05, row, item, transform=label_x, ha="left", va="center", fontsize=10.5, color=INK)
panel.set_xscale("log")
panel.set_xlim(0.04, 14)
panel.text(0.85, heading_y["courteous"], "\u25c2 less", ha="right", va="center", fontsize=9, color=MUTED_INK)
panel.text(1.18, heading_y["courteous"], "more \u25b8", ha="left", va="center", fontsize=9, color=MUTED_INK)
panel.set_ylim(min(row_y.values()) - 0.6, 0.6)
panel.set_yticks([])
panel.set_xticks([])
panel.minorticks_off()
panel.text(1, min(row_y.values()) - 0.75, "1\u00d7 = as often as the median of the rest", ha="center", va="top",
           fontsize=8.5, color=MUTED_INK)
panel.text(-1.12, 1.05, "The first worker compared with the rest of the group\n(\u00d7 the median of the rest of the group)",
           transform=panel.transAxes, ha="left", va="bottom", fontsize=10, color=SECONDARY_INK)
panel.set_facecolor("#fcfcfb")
for side in panel.spines.values():
    side.set_visible(False)
panel.tick_params(length=0)
ax.set_xticks([])
ax.set_yticks([])
for side in ax.spines.values():
    side.set_color("#e1e0d9")
fig.text(0.02, 0.975, "The first worker of the group writes courteously\nand practically, less casually than the rest of us", ha="left", va="top",
         fontsize=17, fontweight="bold", color=INK, linespacing=1.15)
fig.text(0.02, 0.88,
         "Each dot is a stretch of one person's messages; dots close together are written alike.\n"
         "Blue: a friend who started working after high school while the rest of us went on to study.",
         ha="left", va="top", fontsize=10.5, color=SECONDARY_INK, linespacing=1.4)
fig.text(0.02, 0.01,
         f"A dot = 1,500 characters, counted as character trigrams (9 people, 34 dots each); a dot's nearest neighbour is by the same person "
         f"{draws['same author %'].min():.0f}\u2013{draws['same author %'].max():.0f}% of the time over {len(draws)} random draws "
         f"(chance {1 / len(set(authors)):.0%}).\n"
         f"t-SNE (perplexity {DRAFT_PERPLEXITY}) keeps who sits together, not distances: positions and gaps mean nothing. "
         "Links and WhatsApp's edit marker removed.",
         fontsize=7.5, color=MUTED_INK, ha="left")
fig.savefig("who-writes-like-whom-final-draft.png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()